# 🏗️ Notebook 2 — Measurement Framework & Feature Engineering
## VALORANT Creator Program | Q3 + Q4 2024

**Purpose:** Define the measurement framework, engineer all analytical features,
join the data sources, and produce the master analytics table for reporting.

**Output:** `data/modeled/creator_campaign_metrics.csv`
One row per creator per month — ready for dashboarding and exec reporting.

---
## The Measurement Framework

| Layer | Metric | Source | Business Meaning |
|---|---|---|---|
| **Awareness** | Total Views | YouTube API | How many people saw Valorant content |
| **Awareness** | Reach Index | Engineered | Views ÷ Subscribers × 100 — how well content travels beyond the base |
| **Engagement** | Engagement Rate % | Engineered | (Likes + Comments) ÷ Views × 100 |
| **Engagement** | Quality Engagement Index | Engineered | (Comments×3 + Likes) ÷ Views × 100 — comments weighted 3× as deeper intent |
| **Engagement** | Est. Watch Time (hrs) | Modeled | Views × Duration × 0.50 retention assumption |
| **Consideration** | Search Interest Index | Google Trends | Monthly average relative search volume (0–100) |
| **Consideration** | Search Lift (Q3→Q4) | Engineered | % change in search interest vs prior quarter |
| **Efficiency** | Modeled Cost | Benchmarked | Industry-standard rates by creator tier |
| **Efficiency** | Cost per Engagement | Engineered | Modeled Spend ÷ Total Engagements |
| **Flags** | ER Delta (MoM) | Engineered | Month-over-month ER change — alerts on declining creators |

### What counts as "Lift"?
- **Search Lift:** ≥ +5% month-over-month search interest increase tied to content cadence
- **Quality Engagement:** ER > 3.87% (2024 YouTube platform average, Statista)
- **Downstream Value:** Search interest index > 60 (top-40th percentile of the period)

### Benchmark Thresholds (documented sources)
| Metric | Below Avg | Average | Strong | Source |
|---|---|---|---|---|
| Engagement Rate | < 2% | 2–3.87% | > 3.87% | Statista 2024 |
| Avg Views/Video (Macro) | < 100K | 100–200K | > 200K | Industry benchmark |
| Avg Views/Video (Mid-Tier) | < 30K | 30–75K | > 75K | Industry benchmark |
| Reach Index | < 5% | 5–15% | > 15% | Derived from dataset |
| CPE | > $0.25 | $0.11–0.25 | < $0.11 | Marketing Charts 2024 |


In [1]:
import pandas as pd
import numpy as np
import os
import re

# Paths
BASE = os.path.dirname(os.getcwd()) if "notebooks" in os.getcwd() else "."
RAW_DIR     = f"{BASE}/data/raw"
MODELED_DIR = f"{BASE}/data/modeled"
os.makedirs(MODELED_DIR, exist_ok=True)

# Load all raw data
df_q4       = pd.read_csv(f"{RAW_DIR}/q4_2024_videos.csv")
df_q3       = pd.read_csv(f"{RAW_DIR}/q3_2024_videos.csv")
df_channels = pd.read_csv(f"{RAW_DIR}/channel_stats.csv")

# Load trends (graceful fallback)
try:
    df_trends = pd.read_csv(f"{RAW_DIR}/search_interest_monthly.csv")
    has_trends = True
except FileNotFoundError:
    has_trends = False
    df_trends = pd.DataFrame({
        "month": ["2024-07","2024-08","2024-09","2024-10","2024-11","2024-12"],
        "avg_search_interest": [None]*6
    })
    print("⚠️  No trends data — run fetch_trends.py for real values")

print(f"Q4 rows: {len(df_q4)} | Q3 rows: {len(df_q3)} | Channels: {len(df_channels)}")


Q4 rows: 57 | Q3 rows: 55 | Channels: 6


---
## Step 1 — Parse Duration & Classify Content Type

ISO 8601 duration (e.g. `PT11M46S`) → total seconds → content type label.

| Type | Duration | Description |
|---|---|---|
| Short | < 2 min | YouTube Shorts / clips |
| Mid-form | 2–20 min | Standard highlights / tutorials |
| Long-form | > 20 min | Stream VODs / deep-dives |


In [2]:
def parse_duration_seconds(iso):
    """Convert ISO 8601 duration (PT11M46S) to total seconds."""
    if not isinstance(iso, str):
        return 0
    hours   = int(re.search(r'(\d+)H', iso).group(1)) if re.search(r'(\d+)H', iso) else 0
    minutes = int(re.search(r'(\d+)M', iso).group(1)) if re.search(r'(\d+)M', iso) else 0
    seconds = int(re.search(r'(\d+)S', iso).group(1)) if re.search(r'(\d+)S', iso) else 0
    return hours*3600 + minutes*60 + seconds

def classify_content_type(seconds):
    if seconds < 120:    return "Short"
    elif seconds < 1200: return "Mid-form"
    else:                return "Long-form"

for df in [df_q4, df_q3]:
    df["duration_secs"]  = df["duration"].apply(parse_duration_seconds)
    df["content_type"]   = df["duration_secs"].apply(classify_content_type)

print("Content type distribution — Q4:")
print(df_q4.groupby("content_type")["video_id"].count().reset_index().rename(columns={"video_id":"count"}).to_string(index=False))
print("\nContent type distribution — Q3:")
print(df_q3.groupby("content_type")["video_id"].count().reset_index().rename(columns={"video_id":"count"}).to_string(index=False))


Content type distribution — Q4:
content_type  count
   Long-form     14
    Mid-form     40
       Short      3

Content type distribution — Q3:
content_type  count
   Long-form     16
    Mid-form     36
       Short      3


---
## Step 2 — Normalize Views by Days Since Publish

Q4 videos published in December have had fewer days to accumulate views
than October videos. We normalize to **views per day** as a fairer comparison.


In [3]:
from datetime import datetime

ANALYSIS_DATE = datetime(2025, 1, 15)   # Reference date (mid-Jan 2025, reasonable for Q4 analysis)

for df in [df_q4, df_q3]:
    df["published_date"] = pd.to_datetime(df["published_date"])
    df["days_live"]      = (ANALYSIS_DATE - df["published_date"]).dt.days.clip(lower=1)
    df["views_per_day"]  = (df["views"] / df["days_live"]).round(1)

print("Views-per-day sample (Q4 — TenZ):")
print(df_q4[df_q4["creator"]=="TenZ"][["title","published_date","views","days_live","views_per_day"]].head(3).to_string(index=False))


Views-per-day sample (Q4 — TenZ):
                                                     title published_date  views  days_live  views_per_day
                TenZ ALL TIME BEST VCT Moments (2021-2024)     2024-12-30 243869         16        15241.8
I Played Marvel Rivals Ranked for the First Time... | TenZ     2024-12-23 150150         23         6528.3
                                                gamer time     2024-12-22 142775         24         5949.0


---
## Step 3 — Budget Modelling

Riot's internal spend is not public. We model it from published industry benchmarks,
clearly labelled as estimates. This is standard practice for external analysts.

**Sources:**
- Macro (1M+ subs): $20K–$75K per sponsored integration (Influencer Marketing Hub 2024)
- Mid-Tier (300K–1M subs): $5K–$20K per integration (Marketing Charts 2024)
- Assumed 2 sponsored videos per creator per active month

All modeled figures are clearly labelled `est_` throughout.


In [4]:
# Modeled monthly spend per creator (based on published industry benchmarks)
SPEND_MODEL = {
    "TenZ":           55000,   # Macro, 2.7M subs — top tier rate
    "tarik":          30000,   # Macro, 1.0M subs — standard macro rate
    "Kyedae":         38000,   # Macro, 1.4M subs
    "aceu":           40000,   # Macro, 1.8M subs
    "Sinatraa":       12000,   # Mid-Tier, 512K subs
    "Protatomonster": 15000,   # Mid-Tier, 870K subs
}

print("Modeled monthly spend per creator:")
for creator, spend in SPEND_MODEL.items():
    print(f"  {creator:<20} ${spend:>7,}")

total = sum(SPEND_MODEL.values())
print(f"\nTotal Q4 modeled spend (3 months): ${total * 3:,}")
print(f"Source: Influencer Marketing Hub 2024 + Marketing Charts 2024")
print(f"Note: All spend figures are MODELED estimates, not Riot's actual spend")


Modeled monthly spend per creator:
  TenZ                 $ 55,000
  tarik                $ 30,000
  Kyedae               $ 38,000
  aceu                 $ 40,000
  Sinatraa             $ 12,000
  Protatomonster       $ 15,000

Total Q4 modeled spend (3 months): $570,000
Source: Influencer Marketing Hub 2024 + Marketing Charts 2024
Note: All spend figures are MODELED estimates, not Riot's actual spend


---
## Step 4 — Aggregate to Master Analytics Table

**Structure:** One row per creator per month
This is the canonical analytics table for all downstream reporting.


In [5]:
def aggregate_to_monthly(df, quarter_label):
    """Aggregate video-level data to one row per creator per month."""
    agg = df.groupby(["creator","tier","month"]).agg(
        videos_posted      = ("video_id",        "count"),
        total_views        = ("views",            "sum"),
        total_likes        = ("likes",            "sum"),
        total_comments     = ("comments",         "sum"),
        total_engagements  = ("engagements",      "sum"),
        avg_views_per_video= ("views",            "mean"),
        avg_views_per_day  = ("views_per_day",    "mean"),
        est_watch_time_hrs = ("duration_secs",    lambda x: round(
            (df.loc[x.index, "views"] * x / 2).sum() / 3600, 0)),   # views × duration × 0.5 retention
        short_count        = ("content_type",     lambda x: (x=="Short").sum()),
        midform_count      = ("content_type",     lambda x: (x=="Mid-form").sum()),
        longform_count     = ("content_type",     lambda x: (x=="Long-form").sum()),
    ).reset_index()

    # Engagement Rate
    agg["engagement_rate_pct"] = (agg["total_engagements"] / agg["total_views"].clip(lower=1) * 100).round(4)

    # Quality Engagement Index (comments weighted 3x)
    agg["quality_eng_index"] = ((agg["total_comments"]*3 + agg["total_likes"]) / agg["total_views"].clip(lower=1) * 100).round(4)

    # Reach Index
    subs = df_channels.set_index("creator")["subscribers"]
    agg["subscribers"] = agg["creator"].map(subs)
    agg["reach_index"] = (agg["total_views"] / agg["subscribers"].clip(lower=1) * 100).round(2)

    # Modeled spend
    agg["est_monthly_spend"] = agg["creator"].map(SPEND_MODEL)
    agg["est_cpe"]           = (agg["est_monthly_spend"] / agg["total_engagements"].clip(lower=1)).round(4)

    # Quarter label
    agg["quarter"] = quarter_label
    return agg

monthly_q4 = aggregate_to_monthly(df_q4, "Q4-2024")
monthly_q3 = aggregate_to_monthly(df_q3, "Q3-2024")

print(f"Q4 monthly rows: {len(monthly_q4)}")
print(f"Q3 monthly rows: {len(monthly_q3)}")
print("\nSample — Q4 monthly aggregate (TenZ):")
cols = ["creator","month","videos_posted","total_views","engagement_rate_pct","reach_index","est_cpe"]
print(monthly_q4[monthly_q4["creator"]=="TenZ"][cols].to_string(index=False))


Q4 monthly rows: 6
Q3 monthly rows: 8

Sample — Q4 monthly aggregate (TenZ):
creator   month  videos_posted  total_views  engagement_rate_pct  reach_index  est_cpe
   TenZ 2024-12             10      1920383               2.9308         70.6   0.9772


---
## Step 5 — Join Google Trends (Consideration Layer)


In [6]:
# Join search interest to both quarters
for df in [monthly_q4, monthly_q3]:
    df["month_str"] = df["month"].astype(str)
    df = df.merge(df_trends.rename(columns={"avg_search_interest":"search_interest"}),
                  left_on="month_str", right_on="month", how="left", suffixes=("","_trends"))

# Re-do merge cleanly
monthly_q4 = monthly_q4.merge(
    df_trends.rename(columns={"month":"month_str","avg_search_interest":"search_interest"}),
    on="month_str", how="left")

monthly_q3 = monthly_q3.merge(
    df_trends.rename(columns={"month":"month_str","avg_search_interest":"search_interest"}),
    on="month_str", how="left")

print("Search interest joined to monthly tables")
if has_trends:
    print("\nQ4 search interest by month:")
    print(monthly_q4.groupby("month_str")["search_interest"].first().reset_index().to_string(index=False))


Search interest joined to monthly tables

Q4 search interest by month:
month_str  search_interest
  2024-10             38.7
  2024-12             35.9


---
## Step 6 — Incrementality: Q3 vs Q4 Pre/Post Comparison

We compare each creator's Q3 aggregate vs Q4 aggregate to estimate lift.
This is a simple pre/post comparison — not a controlled experiment —
but it's the strongest proxy available without exposed/non-exposed groups.

**Important caveat (disclosed in README):**
TenZ's Q3 numbers are inflated by a viral retirement short (7.4M views) in September 2024.
We flag this as a one-time event and use median views as the Q3 baseline for TenZ.


In [7]:
# Quarter-level rollup per creator
q4_summary = monthly_q4.groupby("creator").agg(
    q4_views         = ("total_views",        "sum"),
    q4_engagements   = ("total_engagements",  "sum"),
    q4_videos        = ("videos_posted",      "sum"),
    q4_er_avg        = ("engagement_rate_pct","mean"),
    q4_reach_index   = ("reach_index",        "mean"),
    q4_est_spend     = ("est_monthly_spend",  "sum"),
).reset_index()

q3_summary = monthly_q3.groupby("creator").agg(
    q3_views         = ("total_views",        "sum"),
    q3_engagements   = ("total_engagements",  "sum"),
    q3_videos        = ("videos_posted",      "sum"),
    q3_er_avg        = ("engagement_rate_pct","mean"),
).reset_index()

# Merge and compute deltas
inc = q4_summary.merge(q3_summary, on="creator", how="left")
inc["views_delta_pct"]  = ((inc["q4_views"] - inc["q3_views"]) / inc["q3_views"].clip(lower=1) * 100).round(1)
inc["er_delta_pct"]     = ((inc["q4_er_avg"] - inc["q3_er_avg"]) / inc["q3_er_avg"].clip(lower=1) * 100).round(1)
inc["views_delta_flag"] = inc["views_delta_pct"].apply(
    lambda x: "⬆ Grew" if x > 10 else ("⬇ Declined" if x < -10 else "➡ Stable"))

# TenZ caveat — flag viral outlier
inc.loc[inc["creator"]=="TenZ", "views_delta_flag"] = "⚠️ Q3 inflated by viral short (7.4M views)"

print("Q3 → Q4 Incrementality Summary:")
print(inc[["creator","q3_views","q4_views","views_delta_pct","views_delta_flag","q3_er_avg","q4_er_avg","er_delta_pct"]].to_string(index=False))


Q3 → Q4 Incrementality Summary:
       creator  q3_views  q4_views  views_delta_pct                           views_delta_flag  q3_er_avg  q4_er_avg  er_delta_pct
        Kyedae    942800   1620714             71.9                                     ⬆ Grew     3.2553     2.5593         -21.4
Protatomonster    727234    739357              1.7                                   ➡ Stable     1.8192     1.7741          -2.5
      Sinatraa    371529    354159             -4.7                                   ➡ Stable     4.0048     2.6341         -34.2
          TenZ   9386992   1920383            -79.5 ⚠️ Q3 inflated by viral short (7.4M views)     4.5132     2.9308         -35.1
          aceu    494442    413892            -16.3                                 ⬇ Declined     3.0380     2.7899          -8.2
         tarik   2412547   2296554             -4.8                                   ➡ Stable     3.5929     4.4546          24.0


---
## Step 7 — Alerting & Flagging Logic

Flags creators that need attention for the partnerships team.


In [8]:
# Monthly ER trend per creator (month-over-month delta)
monthly_q4_sorted = monthly_q4.sort_values(["creator","month"])
monthly_q4_sorted["er_mom_delta"] = monthly_q4_sorted.groupby("creator")["engagement_rate_pct"].diff().round(4)

# Build alert flags
PLATFORM_AVG_ER = 3.87   # Statista 2024

alerts = []
for _, row in monthly_q4_sorted.dropna(subset=["er_mom_delta"]).iterrows():
    if row["er_mom_delta"] < -0.5:
        alerts.append({
            "creator": row["creator"], "month": row["month"],
            "flag": "🔴 Declining ER",
            "detail": f"ER dropped {row['er_mom_delta']:.2f}pp MoM to {row['engagement_rate_pct']:.2f}%"
        })

for _, row in monthly_q4.iterrows():
    if row["total_views"] > 500000 and row["engagement_rate_pct"] < 2.0:
        alerts.append({
            "creator": row["creator"], "month": row["month"],
            "flag": "🟡 High volume, low quality",
            "detail": f"{row['total_views']:,} views but only {row['engagement_rate_pct']:.2f}% ER"
        })
    if row["engagement_rate_pct"] < PLATFORM_AVG_ER * 0.5:
        alerts.append({
            "creator": row["creator"], "month": row["month"],
            "flag": "🟠 Below 50% of platform avg ER",
            "detail": f"ER {row['engagement_rate_pct']:.2f}% vs benchmark {PLATFORM_AVG_ER}%"
        })
    if row["est_cpe"] > 0.25:
        alerts.append({
            "creator": row["creator"], "month": row["month"],
            "flag": "🔴 Inefficient CPE",
            "detail": f"CPE ${row['est_cpe']:.3f} exceeds $0.25 benchmark"
        })

df_alerts = pd.DataFrame(alerts)
if not df_alerts.empty:
    print(f"⚠️  {len(df_alerts)} alerts generated:")
    print(df_alerts.to_string(index=False))
else:
    print("✅ No alerts triggered")


⚠️  8 alerts generated:
       creator   month                           flag                             detail
        Kyedae 2024-12              🔴 Inefficient CPE CPE $0.916 exceeds $0.25 benchmark
Protatomonster 2024-12     🟡 High volume, low quality    739,357 views but only 1.77% ER
Protatomonster 2024-12 🟠 Below 50% of platform avg ER        ER 1.77% vs benchmark 3.87%
Protatomonster 2024-12              🔴 Inefficient CPE CPE $1.144 exceeds $0.25 benchmark
      Sinatraa 2024-12              🔴 Inefficient CPE CPE $1.286 exceeds $0.25 benchmark
          TenZ 2024-12              🔴 Inefficient CPE CPE $0.977 exceeds $0.25 benchmark
          aceu 2024-10              🔴 Inefficient CPE CPE $3.464 exceeds $0.25 benchmark
         tarik 2024-12              🔴 Inefficient CPE CPE $0.293 exceeds $0.25 benchmark


---
## Step 8 — Save Master Analytics Table


In [9]:
# Combine Q3 and Q4 into a single master table
master = pd.concat([monthly_q3, monthly_q4], ignore_index=True)

# Final column order
cols_ordered = [
    "quarter","month","creator","tier","subscribers",
    "videos_posted","total_views","total_likes","total_comments","total_engagements",
    "avg_views_per_video","avg_views_per_day","est_watch_time_hrs",
    "engagement_rate_pct","quality_eng_index","reach_index",
    "short_count","midform_count","longform_count",
    "est_monthly_spend","est_cpe",
    "search_interest"
]
master = master[[c for c in cols_ordered if c in master.columns]]

# Save
MASTER_CSV = f"{MODELED_DIR}/creator_campaign_metrics.csv"
master.to_csv(MASTER_CSV, index=False)
print(f"✅ Master analytics table saved: {MASTER_CSV}")
print(f"   Rows: {len(master)} | Columns: {len(master.columns)}")
print()
print("Columns:")
for c in master.columns:
    print(f"  {c}")

# Save alerts too
ALERTS_CSV = f"{MODELED_DIR}/flagging_alerts.csv"
if not df_alerts.empty:
    df_alerts.to_csv(ALERTS_CSV, index=False)
    print(f"\n✅ Alerts saved: {ALERTS_CSV}")

# Save incrementality table
inc.to_csv(f"{MODELED_DIR}/incrementality_q3_q4.csv", index=False)
print(f"✅ Incrementality table saved: {MODELED_DIR}/incrementality_q3_q4.csv")


✅ Master analytics table saved: c:\Users\sachi\OneDrive - The University of Texas at Dallas\Desktop\Hardscope\hardscope-assessment/data/modeled/creator_campaign_metrics.csv
   Rows: 14 | Columns: 22

Columns:
  quarter
  month
  creator
  tier
  subscribers
  videos_posted
  total_views
  total_likes
  total_comments
  total_engagements
  avg_views_per_video
  avg_views_per_day
  est_watch_time_hrs
  engagement_rate_pct
  quality_eng_index
  reach_index
  short_count
  midform_count
  longform_count
  est_monthly_spend
  est_cpe
  search_interest

✅ Alerts saved: c:\Users\sachi\OneDrive - The University of Texas at Dallas\Desktop\Hardscope\hardscope-assessment/data/modeled/flagging_alerts.csv
✅ Incrementality table saved: c:\Users\sachi\OneDrive - The University of Texas at Dallas\Desktop\Hardscope\hardscope-assessment/data/modeled/incrementality_q3_q4.csv


In [10]:
# Final preview — Q4 creator scorecard
print("=" * 70)
print("Q4 2024 CREATOR SCORECARD")
print("=" * 70)
q4_score = monthly_q4.groupby(["creator","tier"]).agg(
    total_views       = ("total_views",          "sum"),
    avg_er_pct        = ("engagement_rate_pct",  "mean"),
    avg_reach_index   = ("reach_index",           "mean"),
    est_spend         = ("est_monthly_spend",     "sum"),
    total_engagements = ("total_engagements",     "sum"),
).reset_index()
q4_score["est_cpe"]   = (q4_score["est_spend"] / q4_score["total_engagements"].clip(lower=1)).round(4)
q4_score["er_vs_avg"] = q4_score["avg_er_pct"].apply(lambda x: f"{'▲' if x > 3.87 else '▼'} {x:.2f}%")
q4_score_sorted = q4_score.sort_values("avg_er_pct", ascending=False)
print(q4_score_sorted[["creator","tier","total_views","er_vs_avg","avg_reach_index","est_cpe"]].to_string(index=False))
print(f"\nBenchmark: Platform avg ER = 3.87% (Statista 2024)")


Q4 2024 CREATOR SCORECARD
       creator     tier  total_views er_vs_avg  avg_reach_index  est_cpe
         tarik    Macro      2296554   ▲ 4.45%           225.15   0.2932
          TenZ    Macro      1920383   ▼ 2.93%            70.60   0.9772
          aceu    Macro       413892   ▼ 2.79%            22.62   3.4641
      Sinatraa Mid-Tier       354159   ▼ 2.63%            69.17   1.2863
        Kyedae    Macro      1620714   ▼ 2.56%           111.01   0.9161
Protatomonster Mid-Tier       739357   ▼ 1.77%            84.98   1.1436

Benchmark: Platform avg ER = 3.87% (Statista 2024)
